# 🚀 Space Station 14 — Реальная Нативная C# Игра в Браузере (`Content.Client` + `Content.Server`) + ИИ-Экипаж (OpenAI API)

Этот ноутбук разворачивает в **Google Colab** реальную игру **Space Station 14 (`space-wizards/space-station-14`)** прямо в браузере:
1. **Реальный нативный C# OpenGL движок Space Station 14 (`Content.Client` + `Content.Server` с модом `AIStationBridgeSystem`)**:
   - Запускается внутри виртуального X11/OpenGL дисплея (`Xvfb :99` + `Mesa llvmpipe` + `x11vnc`) и транслируется напрямую в браузер (`🖥️ Реальная Игра SS14 (C# LIVE)`) через встроенный WebSocket-мост `/websockify` + `noVNC` на порту `8000`.
   - Вы управляете реальным персонажем в `Content.Client` прямо в браузере (клавиатура `WASD`, `E`, `Z`, `X`, `Q`, `C`, `Tab`, `T` чат, `F7` админ-меню, `G` / `B` сэндбокс и спавнер, консоль `~` -> `aghost` для режима Госта).
2. **Мульти-агентный ИИ-экипаж через ваш OpenAI API (`/v1/chat/completions`)**:
   - Все 14 ИИ-персонажей станции спавнятся прямо внутри реального C# `Content.Server` (`MobHuman`) через наш C# мод `AiStationBridgeSystem` (порт `12120`), считают себя живыми сотрудниками станции NSS Saltern и общаются в игровом чате и по рации.
   - Кнопка **`⚙️ OpenAI API`** в верхней панели позволяет прямо из браузера задать `API Key`, `Base URL` и `Model`.
3. **Монитор камер безопасности (`📹 Камеры СБ`) и быстрый 2D-режим (`🎮 2D Клиент` / `👻 Гост [G]`)**:
   - В любой момент можно переключаться между окном реального C# клиента `Content.Client` и 53 камерами наблюдения СБ / тактическим радаром станции.

In [ ]:
#@title 1️⃣ Клонирование репозитория howlol/ss144 и установка зависимостей
import os, subprocess, sys

REPO_URL = "https://github.com/howlol/ss144.git"
BRANCH = "arena/01a101dd-ss144"
WORK_DIR = "/content/ss144"

if not os.path.exists(WORK_DIR):
    res = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR])
    if res.returncode != 0:
        subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=False)

os.chdir(WORK_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run([
        "bash", "-c",
        "curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
    ], check=True)

print("✅ Репозиторий и зависимости успешно установлены в", os.getcwd())

In [ ]:
#@title 2️⃣ Настройка OpenAI API (можно также менять в любой момент прямо в верхней панели игры — кнопка ⚙️ OpenAI API)
import os

OPENAI_BASE_URL = "https://api.openai.com/v1" #@param {type:"string"}
OPENAI_API_KEY = "" #@param {type:"string"}
OPENAI_MODEL = "gpt-4o-mini" #@param {type:"string"}

os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL.strip()
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
os.environ["OPENAI_MODEL"] = OPENAI_MODEL.strip()
print("✅ Конфигурация OpenAI API сохранена в окружении!")

In [ ]:
#@title 3️⃣ Запуск реальной нативной C# игры Space Station 14 (Content.Server + Content.Client + Xvfb + noVNC) в браузере
import os, re, subprocess, time, urllib.request

subprocess.run(["git", "fetch", "--all"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/arena/01a101dd-ss144"], check=False)
subprocess.run(["pkill", "-f", "uvicorn server.app:app"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(1)

print("🚀 Развёртывание скомпилированного C# движка Space Station 14 (Content.Server + Content.Client + Xvfb + noVNC)...")
stack_proc = subprocess.Popen(
    ["bash", "scripts/run_live_ss14_stack.sh"],
    stdout=open("/tmp/ss14_stack.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(90):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/native_status", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=open("/tmp/cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(1)
    log_txt = open("/tmp/cloudflared.log").read() if os.path.exists("/tmp/cloudflared.log") else ""
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_txt)
    if m:
        public_url = m.group(0)
        break

print("=" * 80)
if public_url:
    print(f"🌐 ОТКРЫТЬ РЕАЛЬНУЮ ИГРУ SPACE STATION 14 В БРАУЗЕРЕ: {public_url}")
print("=" * 80)

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, height="820")
except Exception:
    pass